# Imports & Functions

In [1092]:
import pandas as pd
from pathlib import Path
from ydata_profiling import ProfileReport
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from importlib import reload
import data_loading  # Import the module instead of specific functions


reload(data_loading)  # Reload the module after making changes so that the kernel resets

# reference functions directly from the reloaded module
read_csv_to_dataframe = data_loading.read_csv_to_dataframe
read_txt_to_dataframe = data_loading.read_txt_to_dataframe

In [1093]:
"""
Expands the categories in a given column of a DataFrame into separate binary columns.

Parameters:
- df: pandas.DataFrame, the DataFrame to modify.
- column_name: str, the name of the column to expand.
"""
def expand_categories_in_column(df, column_name):

    # create set of unique entries from a string based on splitting at ", "
    unique_categories = set()
    df[column_name].dropna().apply(lambda x: unique_categories.update(set(x.split(', '))))

    # init dict to hold new columns
    new_cols = {}
    
    for category in unique_categories:
        # add col name as there are duplicates across different cols
        # remove spaces and commas from col name
        valid_category_name = column_name + "_" + category.lower().replace(' ', '_').replace(',', '')
        
        # Instead of modifying df directly, create and store the new column in new_cols
        mask = df[column_name].fillna('').str.contains(category, regex=False, na=False)
        new_cols[valid_category_name] = mask.astype(int)

    # Create a new df from the new_cols dictionary
    new_columns_df = pd.DataFrame(new_cols, index=df.index)
    
    # Concatenate the new columns to the original DataFrame and drop original col
    df = pd.concat([df, new_columns_df], axis=1)
    df = df.drop(columns={column_name})

    return df

In [1094]:
## preprocessing script for logistic regressions dataframes
## based on test_df
def preprocess_df(test_df, unique_col):

    # make a copy, take the appropriate cols, expand accordingly
    df_copy = test_df.copy()
    df_copy = df_copy[['substance carried', unique_col]]
    print(df_copy.count())
    
    df_copy[unique_col] = df_copy[unique_col].str.lower()
    df_copy = expand_categories_in_column(df_copy, unique_col)
    df_copy.drop(columns=[unique_col], inplace=True)
    
    return df_copy

In [1095]:
# correlation matrix and dropping highly-correlated pairs for logistic regression
def handle_correlation_and_drop_columns(test_df):
    # Correlation matrix and identify pairs with correlation > 0.8 (and less than 1)
    corr_matrix = test_df.corr()
    high_corr_pairs = corr_matrix.unstack().sort_values(kind="quicksort", ascending=False)
    high_corr_pairs = high_corr_pairs[(abs(high_corr_pairs) > 0.8) & (high_corr_pairs != 1)]
    
    # Identify columns to drop based on correlations > 0.8
    threshold = 0.8
    to_drop = set()
    for (col1, col2), corr in high_corr_pairs.items():
        if corr > threshold:
            # drop col with less entries
            sum_col1 = test_df[col1].sum()
            sum_col2 = test_df[col2].sum()
            if sum_col1 < sum_col2:
                to_drop.add(col1)
            else:
                to_drop.add(col2)
    
    # Drop cols from above from dataframe
    reduced_df = test_df.drop(columns=list(to_drop))
    return reduced_df

In [1096]:
# logistic regression for qualitative cols
def fit_and_evaluate_logistic_regression(test_df):
    # 'substance carried' is the target variable
    X = test_df.drop(['substance carried'], axis=1)  # Features
    y = test_df['substance carried']  # Target
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    
    # Create and fit the logistic regression model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Predictions and evaluation
    y_pred = model.predict(X_test)
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred), "\n")
    print("Classification report:\n", classification_report(y_test, y_pred), "\n")
    
    # Coefficients
    coefficients = pd.DataFrame(model.coef_.flatten(), index=X.columns, columns=['Coefficient'])
    sorted_coefficients = coefficients.sort_values(by='Coefficient', ascending=False)
    print("Sorted coefficients:\n", sorted_coefficients, "\n")
    
    return model

# Load in dataset A

In [1097]:
## making file path imports more robust

# get directory of current file
current_script_directory = Path.cwd()

# Construct path to data files given relative location
cad_string = current_script_directory / "../data/raw/canada/"
cad_data = cad_string / "pipeline-incidents-comprehensive-data.csv"

# read data into dataframes
CAD_data_raw = read_csv_to_dataframe(cad_data)

File at '/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/src/../data/raw/canada/pipeline-incidents-comprehensive-data.csv' successfully read into a DataFrame.


# Data Cleaning

### Data Dictionary

In [1098]:
# make deep copy for cleaned df
CAD_data_cleaned = CAD_data_raw.copy()

# looked at data dictionary for columns that could be relevant to inherent properties of substance / pipeline specifications / cause of incident
# want to analyze this subset for trends/correlations
## data dict and data don't line up so needed to manually change some; labelled with "## different"
subset_columns = [
"pipeline or facility type",
"pipeline or facility equipment involved",
"rupture",
"incident types", ## different
"conditions that resulted in the operation beyond limits",
"pipeline outside diameter (nps)",
"pipeline length (km)",
"substance carried",
"released substance type",
"facility type", ## different
"facility latitude",
"facility longitude",
"longitude",
"latitude",
"nominal pipe size",
"material",
"material grade",
"schedule",
"design wall thickness (mm)",
"custom design wall thickness (mm)",
"actual wall thickness (mm)",
"licensed maximum operating pressure (kpa)",
"actual operating pressure at time of failure (kpa)",
"year of manufacture",
"most recent cathodic protection reading at incident site (mv vs. cu/cuso4)",
"weld type",
"seam type",
"coating location",
"coating type",
"coating condition",
"application method",
"year when the coating was applied",
"insulation installed",
"detailed what happened", ## diff
"what happened category", ## diff
"detailed why it happened", ## diff
"why it happened category" ## diff
]

# Taking the subset
CAD_data_cleaned = CAD_data_cleaned[subset_columns]

### Substance carried / released substance

In [1099]:
## 3 substance-based columns - if they're all empty/NA then we'll drop the rows as these don't really help us
## EDIT: "substance" and "released substance type" are identical cols so removed substance
CAD_data_cleaned = CAD_data_cleaned.loc[~((CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].isna()))]

In [1100]:
## lots of na's in substance carried col - try to imputate data as best as we can

## OPTION 1: substance carried contains crude oil, released substance does not; released substance = lube oil, drilling fluid, natural gas liquids, diesel fuel, condensate
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()


,released substance type,substance carried
153,Lube Oil,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
232,Drilling Fluid,Crude Oil
284,Natural Gas Liquids,Crude Oil
349,Natural Gas Liquids,"Condensate, Crude Oil, Natural Gas Liquids"
593,Drilling Fluid,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1050,Natural Gas - Sweet,Crude Oil
1160,Diesel Fuel,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1487,Condensate,Crude Oil
1838,Hydraulic Fluid,Crude Oil


In [1101]:
## OPTION 2: substance carried contains crude oil, released subsance does too
## OBSERVATION: released substance type sometimes is more specific (sour vs. sweet) than substance carried
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()


,released substance type,substance carried
10,Crude Oil - Sour,Crude Oil
29,Crude Oil - Sweet,Crude Oil
183,Crude Oil - Synthetic,Crude Oil
196,Crude Oil - Sweet,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."


In [1102]:
## OPTION 3: substance carried doesn't contain crude oil, released substance does? 
## NO RESULTS
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')==False) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried


In [1103]:
## OPTION 4: substance carried doesn't contain crude oil, released substance doesn't either
## HAPPENS A LOT - and the released substance type looks similar as if the substance carried was crude oil
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')==False) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried
4,Natural Gas - Sweet,Natural Gas
14,Jet Fuel,"not applicable, Refined Products-Aviation, Ref..."
16,Natural Gas - Sweet,"Natural Gas, Natural Gas Sweet"
17,Water,White Water
31,Natural Gas - Sour,Natural Gas Sour
79,Natural Gas - Sweet,Natural Gas Sweet
87,Pulp slurry,Sulfite Pulp Slurry
101,Mixed HVP Hydrocarbons,Natural Gas Sweet
281,Contaminated Water,"Natural Gas Sour, Natural Gas Sweet, not appli..."
282,Propane,Natural Gas Sour


In [1104]:
# option 5 - substance carried is na, but released substance contains crude oil
## Can confidently change the substance carried to crude oil based on results from option #3
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil'))][['released substance type', 'substance carried']].drop_duplicates()
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')), 'substance carried'] = 'Crude Oil'

In [1105]:
# option 6 - substance carried is na, released substance does not contain crude oil
## change substance carried to NOT crude oil IFF the released substance type has 0% change of appearing in crude oil (source of list: chatgpt)
not_in_pipeline_crude_oil = [
    "Potassium Hydroxide (caustic solution)",
    "Sulphur Dioxide",
    "Water",
    "Potassium Carbonate",
    "Contaminated Water",
    "Waste Oil",
    "Amine",
    "Produced Water",
    "Glycol",
    "Pulp slurry"
]
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].str.contains('Crude Oil')==False)][['released substance type', 'substance carried']].drop_duplicates()
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].isna()) & (CAD_data_cleaned['released substance type'].isin(not_in_pipeline_crude_oil)), 'substance carried'] = 'NOT crude oil'



In [1106]:
# OPTION 7/8 - released substance type is na; can't assume what was released (if anything); ignore! 
CAD_data_cleaned.loc[(CAD_data_cleaned['substance carried'].str.contains('Crude Oil')) & (CAD_data_cleaned['released substance type'].isna())][['released substance type', 'substance carried']].drop_duplicates()

,released substance type,substance carried
22,NaN,Crude Oil
24,NaN,"Crude Oil, Crude Oil Sour Heavy, Crude Oil Sou..."
1732,NaN,"Condensate, Crude Oil"


In [1107]:
# still left with lots of records where substance carried is N/A
# sadly have to drop these as it's going to be our target/key attribute and we cannot imputate any further
print("dropping {} rows as they don't have a substance carried".format(len(CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].isna()])))
CAD_data_cleaned = CAD_data_cleaned.loc[CAD_data_cleaned['substance carried'].isna()==False]

dropping 396 rows as they don't have a substance carried


### Dropping cols with less than ____ values

In [1108]:
## WANT to work with attributes that have decent data, therefore going to: 
# drop columns that have more than x percentage of na's
# can play around with this value to see how it affects results
threshold = 0.5
CAD_data_cleaned = CAD_data_cleaned.loc[:, CAD_data_cleaned.isnull().mean() < 0.5]

# remaining categorical cols
categorical_cols = CAD_data_cleaned.select_dtypes(include=['object']).columns
categorical_cols

Index(['pipeline or facility type', 'pipeline or facility equipment involved',
       'rupture', 'incident types', 'pipeline outside diameter (nps)',
       'substance carried', 'insulation installed', 'detailed what happened',
       'what happened category', 'detailed why it happened',
       'why it happened category'],
      dtype='object')

In [1109]:
CAD_data_cleaned.head(2)

,pipeline or facility type,pipeline or facility equipment involved,rupture,incident types,pipeline outside diameter (nps),pipeline length (km),substance carried,longitude,latitude,insulation installed,detailed what happened,what happened category,detailed why it happened,why it happened category
4,NaN,Yes,No,Release of Substance,1219.20000000,943.293210,Natural Gas,-108.36036,50.57578,No,"Damage or deterioration mechanism, Constructio...",Defect and Deterioration,"Job or system factors, Inadequate work standar...",Standards and Procedures
10,NaN,Yes,No,Release of Substance,"1219.00000000, 864.00000000, 914.00000000",1248.909872,Crude Oil,-109.80493,52.13984,No,"Damage or deterioration mechanism, Equipment, ...",Equipment Failure,"Job or system factors, Inadequate leadership a...",Inadequate Supervision


### Cleaning numeric

In [1110]:
# only going to take first of pipeline outside diameter values (sometimes provided in different units)
CAD_data_cleaned['pipeline outside diameter (nps)'] = CAD_data_cleaned['pipeline outside diameter (nps)'].str.split(',').str[0].astype(float).fillna(0)

# fill empty length values with 0
CAD_data_cleaned['pipeline length (km)'] = CAD_data_cleaned['pipeline length (km)'].astype(float).fillna(0)

## longitude & latitude look good

### Categorical to Numeric

In [1111]:
## Before exploring the data, want to convert any string-based/categorical columns to binary or numeric
pipeline_map = {'Transmission': 1, 'Processing':2, 'Distribution':3, 'Gathering':4}
CAD_data_cleaned['pipeline or facility type'] = CAD_data_cleaned['pipeline or facility type'].map(pipeline_map).fillna(0)
CAD_data_cleaned['pipeline or facility type'].unique()

array([0., 1., 2., 3., 4.])

In [1112]:
## Generic binary mapping
binary_map = {'Yes': 1, 'No':0}

In [1113]:
# pipeline or facility equipment
CAD_data_cleaned['pipeline or facility equipment involved'] = CAD_data_cleaned['pipeline or facility equipment involved'].map(binary_map)
CAD_data_cleaned['pipeline or facility equipment involved'].unique()

array([1, 0])

In [1114]:
# rupture
CAD_data_cleaned['rupture'] = CAD_data_cleaned['rupture'].map(binary_map)
CAD_data_cleaned['rupture'].unique()

array([0, 1])

In [1115]:
## insulation installed - all equal no so let's just drop; not going to give extra info
CAD_data_cleaned.drop(columns=['insulation installed'], inplace=True)

### Descriptive cols

In [1116]:
## UPDATE ON incident types, detailed what happened, what happened category, detailed why it happened, and why it happened category:

## We really only care about the incidents that were caused/could have been related to the actual substance being carried/released
## THEREFORE - before we expand the columns below, let's FILTER for only the incidents that could have related to the properties of the substance
## Physical properties - heavier oils could take more pressure to push and could also increase wear and tear. Look for: wear and tear, material weardown, pressure, etc.
## Chemical properties - corrosion/cracking, fires due to flammability of substance

## let's work 1-by-1 so that it removes entries (and therefore work) as we go
## starting with the columns that have the least number of classifications - incident types and what happened category

In [1117]:
## incident types
## REMOVING: ["Adverse Environmental Effects", "Operation Beyond Design Limits"]
inc_type_drop = ["Adverse Environmental Effects", "Operation Beyond Design Limits"]
CAD_data_cleaned = CAD_data_cleaned.loc[CAD_data_cleaned['incident types'].isin(inc_type_drop)==False]

In [1118]:
## what happened category
## REMOVING: ["External Interference", "Incorrect Operation", "Natural Force Damage"]
## ACTUALLY: since its a combination of all the options - let's only keep entries that have at least one of the categories that we care about
## WANT TO KEEP: ["Corrosion and Cracking", "Defect and Deterioration", "Equipment Failure", "Other Causes", "To be determined"]

what_happ_keep = ["Corrosion and Cracking", "Defect and Deterioration", "Equipment Failure", "Other Causes", "To be determined"]
CAD_data_cleaned = CAD_data_cleaned.loc[CAD_data_cleaned['what happened category'].apply(lambda x: any(item in x for item in what_happ_keep))]

In [1119]:
len(CAD_data_cleaned)
## already below 400 so don't want to drop many more

361

In [1120]:
## detailed what happened
# every entry includes "damage or deterioration mechanism" or "substandard acts" as a kind of header/categorization
# we don't care about the substandard acts, but we do care about the damage or deterioration
# so let's extract this portion of the string and then ignore the rest

CAD_data_cleaned['detailed what happened'] = CAD_data_cleaned['detailed what happened'].astype(str)

# extract damage portion of string
damage_regex = r"Damage or deterioration mechanism, ([^;]+)"

# clean up matches by removing the original phrase and stripping
def cleanup(matches, phrase):
    cleaned = [match.replace(phrase, '').strip() for match in matches]
    return ';'.join(cleaned)

# Find all occurrences, clean them up, and join with a semi-colon
CAD_data_cleaned['damage_or_deterioration'] = CAD_data_cleaned['detailed what happened'].str.findall(damage_regex).apply(lambda matches: cleanup(matches, "Damage or deterioration mechanism, "))

# drop orig col
CAD_data_cleaned.drop(columns={'detailed what happened'}, inplace=True)

In [1121]:
## why it happened
## really hard to figure out which of these to remove.
## e.g., "failure in communication" maps to three corrosion events.
## siimilarly, "Human Factors" maps to some deterioration events and some equipment failures
## THEREFORE - let's keep all for now

In [1122]:
## detailed why it happened
## similar to above; not going to filter out any

In [1123]:
## incident types - sometimes include multiple options (e.g., release of substance and operation beyond design limits)
## let's expand the unique entries into binary columns and mark them as 1 or 0 based on whether they happened
## (and then drop the original incident types column)
CAD_data_cleaned = expand_categories_in_column(CAD_data_cleaned, 'incident types')

# can do the same for what happened category as it has minimal entries
## reminder that "natural force" will still exist (among others) as they happened in tandem with corrosion, etc.
CAD_data_cleaned = expand_categories_in_column(CAD_data_cleaned, 'what happened category')

In [1124]:
## detailed what happened - now "damage_or_deterioration" - unsure if we should expand this? 
# sorted(CAD_data_cleaned['damage_or_deterioration'].astype(str).unique())

## same thought process for "detailed why it happened" and "why it happened category"
## too many options and may be providing repetitive data
CAD_data_cleaned.drop(columns={'damage_or_deterioration', 'detailed why it happened', 'why it happened category'}, inplace=True)

# CAD_data_cleaned = expand_categories_in_column(CAD_data_cleaned, 'damage_or_deterioration')
# CAD_data_cleaned = expand_categories_in_column(CAD_data_cleaned, 'detailed why it happened')
# CAD_data_cleaned = expand_categories_in_column(CAD_data_cleaned, 'why it happened category')

In [1125]:
## substance carried will be analysis specific so will change it then

# Crude vs. Not Crude

### Data Exploration

In [1126]:
## want to update substance type so that it's either "Crude" or "not crude" so that we can more easily identify trends
all_cad = CAD_data_cleaned.copy()

# LET's map non crude to 0 and crude to 1
all_cad.loc[all_cad['substance carried'].astype(str).str.contains('Crude Oil')==False, 'substance carried'] = 0

## FOR argument's sake, let's also combine the crude oils
all_cad.loc[all_cad['substance carried'] != 0 , 'substance carried'] = 1

In [1127]:
## GENERATE a profiling report to identify any preliminary trends/issues in the data

# drop duplicates (discussed in results below)
all_cad = all_cad.drop_duplicates(keep='first')

# Define the path to the report file
report_path = Path("../reports/dataset_A_all_cad_profile_report.html")

# only generate report if it doesn't already exist
if not report_path.exists():
    all_cad_profile = ProfileReport(all_cad, title="All CAD data Profiling Report", explorative=True)
    all_cad_profile.to_file(report_path)
else:
    print("Report already exists.")

## RESULTS
# duplicate rows - manual inspection of data shows that these were identical incidents reported by different people.
# let's drop duplicate and keep first even though some of them have different attribute values
# doing this above and then rerunning
    
## HEATMAP / CORRELATION
# releases of substances are highly correlated with fires
# pipeline length is highly correlated with pipeline outside diameter
## SUBSTANCE CARRIED is highly correlated with latitude (and pipeline length)
# corrosion and cracking highly correlated with equipment failure


Report already exists.


In [1128]:
# SPLIT df into substance carried contains crude oil or does not so that we can get quantitative descriptive statistics (qualitative in html file)
crude_cad = all_cad.copy()
crude_cad = crude_cad.loc[crude_cad['substance carried'] == 1]

non_crude_cad = all_cad.copy()
non_crude_cad = non_crude_cad.loc[non_crude_cad['substance carried'] == 0]

In [1129]:
non_crude_cad.describe()

,pipeline or facility type,pipeline or facility equipment involved,rupture,pipeline outside diameter (nps),pipeline length (km),longitude,latitude,incident types_serious_injury_(cer_or_tsb),incident types_explosion,incident types_release_of_substance,incident types_adverse_environmental_effects,incident types_fire,what happened category_defect_and_deterioration,what happened category_natural_force_damage,what happened category_other_causes,what happened category_to_be_determined,what happened category_equipment_failure,what happened category_external_interference,what happened category_corrosion_and_cracking,what happened category_incorrect_operation
count,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.000000,278.00000,278.000000,278.000000,278.000000,278.000000,278.000000
mean,0.949640,0.852518,0.043165,510.826619,614.477352,-100.373835,51.306148,0.003597,0.014388,0.852518,0.021583,0.154676,0.208633,0.014388,0.05036,0.017986,0.399281,0.122302,0.366906,0.125899
std,0.885697,0.355225,0.203596,323.226460,974.054922,21.623142,4.315991,0.059976,0.119301,0.355225,0.145579,0.362248,0.407064,0.119301,0.21908,0.133139,0.490634,0.328225,0.482830,0.332334
min,0.000000,0.000000,0.000000,0.000000,0.000000,-122.689900,42.287869,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,273.100000,16.596023,-118.628477,47.361386,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,0.000000,508.000000,112.455223,-111.685646,51.669793,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,762.000000,574.880799,-79.498208,55.294618,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,1.000000,0.000000,1.000000,0.000000
max,4.000000,1.000000,1.000000,1362.400000,3393.426420,-61.612676,58.655000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000


In [1130]:
crude_cad.describe()

,pipeline or facility type,pipeline or facility equipment involved,rupture,pipeline outside diameter (nps),pipeline length (km),longitude,latitude,incident types_serious_injury_(cer_or_tsb),incident types_explosion,incident types_release_of_substance,incident types_adverse_environmental_effects,incident types_fire,what happened category_defect_and_deterioration,what happened category_natural_force_damage,what happened category_other_causes,what happened category_to_be_determined,what happened category_equipment_failure,what happened category_external_interference,what happened category_corrosion_and_cracking,what happened category_incorrect_operation
count,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.000000,74.00000
mean,0.851351,0.743243,0.013514,521.231081,753.365676,-111.154232,51.074436,0.013514,0.054054,0.635135,0.013514,0.310811,0.148649,0.040541,0.081081,0.013514,0.689189,0.310811,0.162162,0.22973
std,0.974784,0.439826,0.116248,432.604686,642.472629,9.883003,2.807402,0.116248,0.227668,0.484678,0.116248,0.465985,0.358170,0.198569,0.274823,0.116248,0.465985,0.465985,0.371116,0.42353
min,0.000000,0.000000,0.000000,0.000000,0.000000,-122.950276,42.952425,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,0.000000,0.250000,0.000000,0.000000,0.000000,-120.233745,49.268490,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,1.000000,1.000000,0.000000,559.000000,946.159155,-113.351635,50.483397,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.00000
75%,1.000000,1.000000,0.000000,762.000000,1255.999121,-104.593327,53.357191,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.00000
max,4.000000,1.000000,1.000000,1219.000000,2334.948756,-77.929450,61.861680,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


### Logistic Regression

In [1131]:
## GET JACK'S HELP HERE - given my dataframe, what can i do to see if all of the variables have any impact/correlation to the substance carried? 

In [1132]:
# # preprocess test df into 3 dfs each with the respective categorical col expanded
# test_df_1 = preprocess_df(test_df, 'what happened category')
# test_df_2 = preprocess_df(test_df, 'detailed what happened')
# test_df_3 = preprocess_df(test_df, 'incident types')

In [1133]:
# correlation/preprocessing
# test_df_1_reduced = handle_correlation_and_drop_columns(test_df_1)
# test_df_2_reduced = handle_correlation_and_drop_columns(test_df_2)
# test_df_3_reduced = handle_correlation_and_drop_columns(test_df_3)
all_cad_red = handle_correlation_and_drop_columns(all_cad)
all_cad_red['substance carried'] = all_cad_red['substance carried'].astype(int)

print("Results:")
model_2 = fit_and_evaluate_logistic_regression(all_cad_red)

# print("Results for 'incident types'")
# model_3 = fit_and_evaluate_logistic_regression(test_df_3_reduced)

# # run regression model on each df
# print("Results for 'what happened category'")
# model_1 = fit_and_evaluate_logistic_regression(test_df_1_reduced)

# print("Results for 'detailed what happened'")
# model_2 = fit_and_evaluate_logistic_regression(test_df_2_reduced)

# print("Results for 'incident types'")
# model_3 = fit_and_evaluate_logistic_regression(test_df_3_reduced)

Results:
Confusion matrix:
 [[64  5]
 [15  4]] 

Classification report:
               precision    recall  f1-score   support

           0       0.81      0.93      0.86        69
           1       0.44      0.21      0.29        19

    accuracy                           0.77        88
   macro avg       0.63      0.57      0.58        88
weighted avg       0.73      0.77      0.74        88
 

Sorted coefficients:
                                                  Coefficient
what happened category_equipment_failure            0.807899
what happened category_external_interference        0.752820
what happened category_natural_force_damage         0.708026
what happened category_other_causes                 0.628479
what happened category_to_be_determined             0.417010
pipeline or facility equipment involved             0.378511
incident types_explosion                            0.375570
incident types_serious_injury_(cer_or_tsb)          0.241625
incident types_release_of_s

/Users/cameronmackinnon/PD/Winter 2024 Job Hunt/Companies/Validere/crude_oil_accidents/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
## CONCLUSION - none are great predictors of crude oil vs. not
## 3 qualitative cols obtained similar outcomes
## this could mean that crude oil vs. not have similar accident patterns, AKA, inherent properties of crude oil (relative) to other substances don't have an affect on pipeline accidents

# Crude Only

In [966]:
## START with cleaned dataframe
cad_crude = CAD_data_cleaned.copy()
#cad_crude['substance carried'].unique()

# only want to look at crude oil incidents
cad_crude = cad_crude.loc[(cad_crude['substance carried'].str.contains('Crude Oil')) | (cad_crude['released substance type'].str.contains('Crude Oil'))]

# now let's extract the specifications (sour, sweet, heavy, light) into other columns so they're easier to deal with

#cad_crude[['substance carried', 'released substance type']].drop_duplicates()
cad_crude['substance carried'].unique()

## ERROR - i thought it was separate - i.e., crude oil sour, crude oil sweet, but it's actually all or nothing, i.e., crude oil or crude oil sour, sweet, heavy, light
## so this sadly doesn't help our analysis :(

## ONTO dataset B

array(['Crude Oil',
       'Crude Oil, Crude Oil Sour Heavy, Crude Oil Sour Light, Crude Oil Sweet Heavy, Crude Oil Sweet Light',
       'Condensate, Crude Oil, Natural Gas Liquids',
       'Condensate, Crude Oil'], dtype=object)

# unused code

In [ ]:
# ## UPDATE - chem words slims the data down too much. not going to use for now. 

## LOTS OF entries in the qualitative columns - so trying to minimize this by focusing on things that could be related to the substance
## went through for words related to chemistry - temperatures, corrosion, cracking, etc.
## filter first for entries that contain these, make sure lower case
# chem_words = ['corrosion', 'temperature', 'deterioration', 'overheating', 'weather', 'frost', 'fire', 'frozen', 'chemical', 'cracking']

# def contains_chem_words(text, chem_words):
#     if pd.isna(text):
#         return False
#     return any(chem_word in text for chem_word in chem_words)

# Create a boolean mask where at least one of the conditions is True
# mask = test_df.apply(lambda row: contains_chem_words(row['what happened category'], chem_words) or 
#                                  contains_chem_words(row['detailed what happened'], chem_words), axis=1)

# mask = test_df.apply(lambda row: contains_chem_words(row['what happened category'], chem_words), axis=1)

# # Filter the DataFrame
# filtered_df = test_df[mask]

In [ ]:
# want to look at incidents that could have been influenced by the material
# e.g., corrosion/cracking
# hard to determine... leaving this for now
# reasons_df = CAD_data_cleaned.copy()[['Detailed what happened', 'What happened category', 'Detailed why it happened', 'Why it happened category']].drop_duplicates()
# reasons_df['Why it happened category'].unique()

## columns that could be related to material influencing an accident/corrosion

#print(CAD_data_cleaned['Pipeline or Facility Type'].unique())
#CAD_data_cleaned['Rupture'].unique() ## loss of containment and unable to operate
#CAD_data_cleaned['Regulation'].unique()
#print(CAD_data_cleaned['Facility Type'].unique())

In [ ]:
# removing inches and keeping in mm
#CAD_data_cleaned['design wall thickness (mm)'] = CAD_data_cleaned['design wall thickness (mm)'].str.split(' mm').str[0].astype(float).fillna(0)